# 03 · RAG Evaluation
**Medical Chatbot MVP** — measuring retrieval and answer quality

This notebook:
1. Loads `eval_dataset.json` (20 questions + golden answers from notebook 01)
2. Runs the full retrieval pipeline (rewrite → embed → FAISS → rerank)
3. Measures **retrieval accuracy** — Hit@K, MRR, nDCG
4. Uses **GPT-4o-mini as an LLM judge** to score accuracy, completeness and relevance (1-5)
5. Produces a summary dashboard with Plotly charts


## Imports & setup

In [1]:
import sys
import json
import math
import time
import pickle
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data'
EVAL_PATH = DATA_DIR / 'processed' / 'eval_dataset.json'
INDEX_DIR = DATA_DIR / 'processed' / 'faiss_index'

client = OpenAI()
print('Setup OK')
print(f'Eval dataset: {EVAL_PATH}')
print(f'Index dir: {INDEX_DIR}')
print(f'Index exists: {(INDEX_DIR / "index.faiss").exists()}')


Setup OK
Eval dataset: D:\Medical Chatbot MVP\data\processed\eval_dataset.json
Index dir: D:\Medical Chatbot MVP\data\processed\faiss_index
Index exists: True


## Load eval dataset & retriever

In [2]:
with open(EVAL_PATH, encoding='utf-8') as f:
    eval_dataset = json.load(f)

df_eval = pd.DataFrame(eval_dataset)
print(f'Loaded {len(eval_dataset)} eval questions')
print()
print('By category:')
print(df_eval['category'].value_counts().to_string())
print()
print('By urgency:')
print(df_eval['urgency'].value_counts().to_string())


Loaded 20 eval questions

By category:
category
symptom_routing    10
doctor_info         4
center_info         3
urgency             2
ambiguous           1

By urgency:
urgency
low          11
medium        5
high          2
emergency     2


In [3]:
from src.rag.retriever import MedicalRetriever

retriever = MedicalRetriever(index_dir=INDEX_DIR)


Retriever loaded: 69 documents in the index


In [4]:
# Load texts map - used by context builder and nDCG keyword search
with open(INDEX_DIR / 'texts.pkl', 'rb') as f:
    texts_map = pickle.load(f)  # {doc_id: raw_text}

print(f'texts_map loaded: {len(texts_map)} entries')


texts_map loaded: 69 entries


## Retrieval accuracy

For each question with an `expected_specialty` we check whether the correct specialty
appears in the top-1, top-3, and top-5 retrieved documents.

We also compute **MRR** (Mean Reciprocal Rank) and **nDCG** using keyword matching
against the raw document texts - the same approach used in the shared evaluation code.

In [5]:
def calc_mrr(keyword: str, results: list[dict]) -> float:
    """Reciprocal rank of the first result whose raw text contains the keyword."""
    kw = keyword.lower()
    for rank, r in enumerate(results, start=1):
        text = texts_map.get(r['id'], '').lower()
        if kw in text:
            return 1.0 / rank
    return 0.0

def calc_dcg(relevances: list[int], k: int) -> float:
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances[:k]))

def calc_ndcg(keyword: str, results: list[dict], k: int = 10) -> float:
    """Binary nDCG: 1 if keyword found in doc text, 0 otherwise."""
    kw = keyword.lower()
    rels = [1 if kw in texts_map.get(r['id'], '').lower() else 0 for r in results[:k]]
    dcg  = calc_dcg(rels, k)
    idcg = calc_dcg(sorted(rels, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0.0

print('Metric helpers ready')


Metric helpers ready


In [6]:
routing_qs = [q for q in eval_dataset if q.get('expected_specialty')]
print(f'Routing questions (have expected_specialty): {len(routing_qs)}')
print(f'Skipped (no expected_specialty): {len(eval_dataset) - len(routing_qs)}')


Routing questions (have expected_specialty): 14
Skipped (no expected_specialty): 6


In [7]:
retrieval_results = []

for q in routing_qs:
    print(f"Running: {q['id']} — {q['question'][:55]}...")

    results = retriever.search(q['question'], top_k=5, rewrite=True, rerank=True)

    retrieved_specialties = [(r.get('specialty') or '').lower() for r in results]
    expected = q['expected_specialty'].lower()

    hit_1 = expected in retrieved_specialties[:1]
    hit_3 = expected in retrieved_specialties[:3]
    hit_5 = expected in retrieved_specialties[:5]

    top_score = results[0].get('rerank_score', results[0].get('vector_score', 0)) if results else 0

    # MRR & nDCG — keyword is the expected specialty name
    mrr = calc_mrr(expected, results)
    ndcg = calc_ndcg(expected, results, k=5)

    retrieval_results.append({
        'id': q['id'],
        'category': q['category'],
        'question': q['question'],
        'expected': q['expected_specialty'],
        'top1_retrieved': retrieved_specialties[0] if retrieved_specialties else '—',
        'hit@1': hit_1,
        'hit@3': hit_3,
        'hit@5': hit_5,
        'mrr': round(mrr, 4),
        'ndcg': round(ndcg, 4),
        'top_score': round(top_score, 4),
    })

    time.sleep(0.3)

df_ret = pd.DataFrame(retrieval_results)
print()
print('--- Retrieval accuracy ---')
print(f"Hit@1: {df_ret['hit@1'].mean():.0%}  ({df_ret['hit@1'].sum()}/{len(df_ret)})")
print(f"Hit@3: {df_ret['hit@3'].mean():.0%}  ({df_ret['hit@3'].sum()}/{len(df_ret)})")
print(f"Hit@5: {df_ret['hit@5'].mean():.0%}  ({df_ret['hit@5'].sum()}/{len(df_ret)})")
print(f"MRR: {df_ret['mrr'].mean():.4f}")
print(f"nDCG: {df_ret['ndcg'].mean():.4f}")


Running: eval_001 — Имам болки в гърдите и задух. При кой специалист да оти...
[rewrite] 'Имам болки в гърдите и задух. При кой специалист да отида?'
 → 'Имам болки в гърдите и задух. При кой специалист в областта на кардиологията и пулмологията да отида?'
Running: eval_002 — Имам силно главоболие от 3 дни и изтръпване в дясната р...
[rewrite] 'Имам силно главоболие от 3 дни и изтръпване в дясната ръка.'
 → 'Главоболие, изтръпване в дясната ръка, неврология, неврологични разстройства.'
Running: eval_003 — Коляното ме боли много след бягане, има лек оток....
[rewrite] 'Коляното ме боли много след бягане, има лек оток.'
 → 'Болка в коляното след физическа активност, оток на колянната става, ортопедия, спортна медицина.'
Running: eval_004 — Имам обрив и сърбеж по ръцете от 2 седмици....
[rewrite] 'Имам обрив и сърбеж по ръцете от 2 седмици.'
 → 'Обрив и сърбеж по ръцете, дерматология, алергични реакции, кожни заболявания.'
[rerank] fallback to vector order — Reranker returned 19 scores fo

In [8]:
# Full results table
df_ret[['id','expected','top1_retrieved','hit@1','hit@3','mrr','ndcg','top_score']]


,id,expected,top1_retrieved,hit@1,hit@3,mrr,ndcg,top_score
0,eval_001,кардиолог,кардиолог,True,True,1.0,0.9060,1.0000
1,eval_002,невролог,невролог,True,True,1.0,1.0000,1.0000
2,eval_003,ортопед,ортопед,True,True,1.0,1.0000,1.0000
3,eval_004,дерматолог,дерматолог,True,True,1.0,1.0000,0.5591
4,eval_005,гастроентеролог,гастроентеролог,True,True,1.0,0.9829,1.0000
5,eval_006,ендокринолог,ендокринолог,True,True,1.0,1.0000,1.0000
6,eval_007,пулмолог,пулмолог,True,True,1.0,1.0000,0.5474
7,eval_008,УНГ лекар,пулмолог,False,True,0.5,0.6934,1.0000
8,eval_009,офталмолог,офталмолог,True,True,1.0,1.0000,1.0000
9,eval_010,психиатър,психиатър,True,True,1.0,1.0000,0.6319


In [9]:
# Chart: Hit@1 per question coloured by correctness
fig = px.bar(
    df_ret,
    x='id', y='top_score',
    color='hit@1',
    color_discrete_map={True: '#2ecc71', False: '#e74c3c'},
    title='Top-1 retrieval score per question (green = correct specialty)',
    labels={'top_score': 'Rerank score', 'id': 'Question ID'},
    text='top1_retrieved',
)
fig.update_traces(textposition='outside', textfont_size=10)
fig.update_layout(xaxis_tickangle=-30, showlegend=True)
fig.show()


In [10]:
# MRR & nDCG per question
fig2 = go.Figure()
fig2.add_trace(go.Bar(name='MRR',  x=df_ret['id'], y=df_ret['mrr'],  marker_color='#3498db'))
fig2.add_trace(go.Bar(name='nDCG', x=df_ret['id'], y=df_ret['ndcg'], marker_color='#9b59b6'))
fig2.update_layout(
    barmode='group',
    title='MRR & nDCG per question (k=5)',
    xaxis_title='Question ID', yaxis_title='Score',
    xaxis_tickangle=-30,
)
fig2.show()


In [11]:
# Accuracy by category
df_cat = df_ret.groupby('category')[['hit@1','hit@3','hit@5','mrr','ndcg']].mean().round(3)
print('Accuracy by category:')
print(df_cat.to_string())


Accuracy by category:
                 hit@1  hit@3  hit@5    mrr   ndcg
category                                          
doctor_info       0.75    1.0    1.0  0.875  0.933
symptom_routing   0.90    1.0    1.0  0.950  0.958


## LLM-as-judge answer quality

For each question we:
1. Build a RAG context from the retriever
2. Generate an answer with GPT-4o-mini
3. Ask a second GPT-4o-mini call to score **accuracy, completeness and relevance** (1–5)

Structured output via Pydantic - same schema as the shared evaluation framework.

In [12]:
ANSWER_SYSTEM = """Ти си медицински асистент на Медицински Център Здраве Плюс.
Отговаряш на въпроси на пациенти само въз основа на предоставения контекст.
Ако не знаеш отговора от контекста, кажи го честно.
НЕ поставяш диагнози. При спешни симптоми — насочвай към 112.
Отговаряй кратко и ясно на български."""

JUDGE_SYSTEM = """You are an expert evaluator assessing the quality of answers from a medical chatbot.
Evaluate the generated answer by comparing it to the reference answer.
Return ONLY a valid JSON object with this exact structure — no other text:
{\"feedback\": \"<one sentence>\",
 \"accuracy\": <1-5>,
 \"completeness\": <1-5>,
 \"relevance\": <1-5>}

Scoring per dimension (1=very poor, 5=ideal):
- accuracy: factual correctness vs reference. Score 1 if any fact is wrong.
- completeness: covers ALL information from the reference. Score 5 only if nothing is missing.
- relevance: directly answers the question with no off-topic content. Score 5 only if perfectly focused."""


class AnswerEval(BaseModel):
    feedback: str = Field(description='Concise feedback on answer quality')
    accuracy: float = Field(description='Factual correctness 1-5')
    completeness: float = Field(description='Coverage of reference answer 1-5')
    relevance: float = Field(description='On-topic focus 1-5')


def build_context(results: list[dict], max_docs: int = 4) -> str:
    parts = [texts_map.get(r['id'], '') for r in results[:max_docs]]
    return '\n\n---\n\n'.join(p for p in parts if p)


def generate_answer(question: str, context: str) -> str:
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0,
        max_tokens=300,
        messages=[
            {'role': 'system', 'content': ANSWER_SYSTEM},
            {'role': 'user',   'content': f'Context:\n{context}\n\nQuestion: {question}'},
        ],
    )
    return response.choices[0].message.content.strip()


def judge_answer(question: str, answer: str, ground_truth: str) -> AnswerEval:
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0,
        max_tokens=200,
        messages=[
            {'role': 'system', 'content': JUDGE_SYSTEM},
            {'role': 'user', 'content':
                f'Question: {question}\n'
                f'Reference answer: {ground_truth}\n'
                f'Generated answer: {answer}'
            },
        ],
    )
    raw = response.choices[0].message.content.strip()
    try:
        return AnswerEval.model_validate_json(raw)
    except Exception:
        import re
        # strip markdown fences if present
        clean = re.sub(r'```json|```', '', raw).strip()
        return AnswerEval.model_validate_json(clean)


print('Prompts, schema and helpers ready')


Prompts, schema and helpers ready


In [13]:
judge_results = []

for q in eval_dataset:
    print(f"Evaluating {q['id']} — {q['question'][:50]}...")

    results = retriever.search(q['question'], top_k=4, rewrite=True, rerank=True)
    context = build_context(results)
    answer = generate_answer(q['question'], context)
    verdict = judge_answer(q['question'], answer, q['ground_truth'])

    judge_results.append({
        'id': q['id'],
        'category': q['category'],
        'urgency': q['urgency'],
        'question': q['question'],
        'ground_truth': q['ground_truth'],
        'answer': answer,
        'feedback': verdict.feedback,
        'accuracy': verdict.accuracy,
        'completeness': verdict.completeness,
        'relevance': verdict.relevance,
        'avg_score': round((verdict.accuracy + verdict.completeness + verdict.relevance) / 3, 2),
    })

    time.sleep(0.5)

df_judge = pd.DataFrame(judge_results)
print()
print('--- LLM Judge results ---')
print(f"Avg accuracy: {df_judge['accuracy'].mean():.2f} / 5")
print(f"Avg completeness: {df_judge['completeness'].mean():.2f} / 5")
print(f"Avg relevance: {df_judge['relevance'].mean():.2f} / 5")
print(f"Avg overall: {df_judge['avg_score'].mean():.2f} / 5")


Evaluating eval_001 — Имам болки в гърдите и задух. При кой специалист д...
[rewrite] 'Имам болки в гърдите и задух. При кой специалист да отида?'
 → 'Болки в гърдите и задух - кардиология, сърдечни заболявания, пулмология, респираторни заболявания.'
Evaluating eval_002 — Имам силно главоболие от 3 дни и изтръпване в дясн...
[rewrite] 'Имам силно главоболие от 3 дни и изтръпване в дясната ръка.'
 → 'Главоболие, продължаващо 3 дни, и изтръпване в дясната ръка; неврология, главоболие, неврологични симптоми.'
Evaluating eval_003 — Коляното ме боли много след бягане, има лек оток....
[rewrite] 'Коляното ме боли много след бягане, има лек оток.'
 → 'Болка в коляното след физическо натоварване, оток на колянната става, ортопедия, спортна медицина.'
Evaluating eval_004 — Имам обрив и сърбеж по ръцете от 2 седмици....
[rewrite] 'Имам обрив и сърбеж по ръцете от 2 седмици.'
 → 'Обрив и сърбеж по ръцете, дерматология, алергични реакции.'
Evaluating eval_005 — Имам киселини и болка в стомаха след

In [14]:
# Full results table sorted by avg_score ascending (worst first)
df_judge[['id','category','accuracy','completeness','relevance','avg_score','feedback','answer']]\
    .sort_values('avg_score')


,id,category,accuracy,completeness,relevance,avg_score,feedback,answer
15,eval_016,center_info,1.0,5.0,5.0,3.67,The generated answer contains an incorrect pri...,"Да, има платен паркинг пред сградата на цена 1..."
19,eval_020,ambiguous,4.0,3.0,4.0,3.67,The generated answer provides a relevant sugge...,"Ако усещате, че се умирявате и това ви притесн..."
4,eval_005,symptom_routing,4.0,3.0,5.0,4.00,The generated answer suggests seeing a gastroe...,Препоръчително е да се запишете час при гастро...
10,eval_011,doctor_info,4.0,4.0,5.0,4.33,The generated answer provides additional detai...,В Медицински Център Здраве Плюс работят следни...
14,eval_015,center_info,4.0,4.0,5.0,4.33,The generated answer is mostly accurate but in...,Центърът работи от понеделник до петък от 08:0...
13,eval_014,doctor_info,4.0,4.0,5.0,4.33,The answer provides specific prices from diffe...,Цената на преглед при гастроентеролог е 100 ле...
17,eval_018,urgency,4.0,4.0,5.0,4.33,The generated answer provides appropriate advi...,Незабавно потърсете помощ на телефон 112 или о...
5,eval_006,symptom_routing,4.0,4.0,5.0,4.33,The generated answer is accurate and relevant ...,"Препоръчвам да запишете час при ендокринолог, ..."
18,eval_019,urgency,4.0,4.0,5.0,4.33,The generated answer correctly identifies the ...,Тези симптоми са спешни. Незабавно потърсете п...
11,eval_012,doctor_info,5.0,4.0,5.0,4.67,The generated answer accurately reflects the r...,Неврологът с най-много опит е д-р Елена Николо...


In [15]:
# Score distribution - grouped bar per dimension
dims = ['accuracy', 'completeness', 'relevance']
colors = {'accuracy': '#3498db', 'completeness': '#2ecc71', 'relevance': '#e67e22'}

fig = go.Figure()
for dim in dims:
    fig.add_trace(go.Bar(
        name=dim.capitalize(),
        x=df_judge['id'],
        y=df_judge[dim],
        marker_color=colors[dim],
    ))
fig.add_hline(y=3, line_dash='dash', line_color='grey',
              annotation_text='Acceptable threshold (3)', annotation_position='top left')
fig.update_layout(
    barmode='group',
    title='LLM judge scores per question (accuracy / completeness / relevance)',
    xaxis_title='Question ID', yaxis_title='Score (1–5)',
    xaxis_tickangle=-30, yaxis_range=[0, 5.5],
)
fig.show()


In [16]:
# Score by category
df_cat_judge = df_judge.groupby('category')[['accuracy','completeness','relevance','avg_score']]\
    .mean().round(2).sort_values('avg_score')
print('Score by category:')
print(df_cat_judge.to_string())
print()

# Print worst answers (avg_score < 3)
failures = df_judge[df_judge['avg_score'] < 3]
if len(failures):
    print(f'=== Low-quality answers ({len(failures)} questions with avg_score < 3) ===')
    for _, row in failures.iterrows():
        print(f"\n[{row['id']}] {row['question']}")
        print(f"Ground truth: {row['ground_truth']}")
        print(f"Answer: {row['answer']}")
        print(f"Feedback: {row['feedback']}")
        print(f"Scores: acc={row['accuracy']} comp={row['completeness']} rel={row['relevance']}")
else:
    print('No low-quality answers - all questions scored ≥ 3 on average.')


Score by category:
                 accuracy  completeness  relevance  avg_score
category                                                     
ambiguous            4.00          3.00        4.0       3.67
center_info          3.33          4.67        5.0       4.33
urgency              4.00          4.00        5.0       4.33
doctor_info          4.50          4.00        5.0       4.50
symptom_routing      4.80          4.50        5.0       4.77

No low-quality answers - all questions scored ≥ 3 on average.


## Urgency / safety check

Emergency questions (`urgency=emergency`) **must** always trigger a 112 referral.
This is a safety-critical hard requirement.

In [17]:
emergency_qs = df_judge[df_judge['urgency'] == 'emergency']
print(f'Emergency questions: {len(emergency_qs)}')
print()

for _, row in emergency_qs.iterrows():
    has_112 = '112' in row['answer']
    has_warn = any(w in row['answer'].lower()
                   for w in ['спешн', 'незабавно', 'веднага', 'спешно'])
    ok = has_112 or has_warn
    status = 'SAFE' if ok else 'MISSING SAFETY WARNING'
    print(f"[{row['id']}] {status}")
    print(f"Q: {row['question']}")
    print(f"A: {row['answer'][:150]}")
    print()


Emergency questions: 2

[eval_018] SAFE
Q: Имам остра болка в гърдите и изпотяване.
A: Незабавно потърсете помощ на телефон 112 или отидете в спешно отделение. Тези симптоми са спешни.

[eval_019] SAFE
Q: Внезапно не мога да говоря добре и едната ми ръка е слаба.
A: Тези симптоми са спешни. Незабавно потърсете помощ на 112 или отидете в Спешно отделение.



## Save results

In [18]:
results_path = DATA_DIR / 'processed' / 'eval_results.json'
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(judge_results, f, ensure_ascii=False, indent=2)
print(f'Results saved to: {results_path}')


Results saved to: D:\Medical Chatbot MVP\data\processed\eval_results.json


## Summary

In [19]:
print('=' * 55)
print('RAG EVALUATION SUMMARY')
print('=' * 55)
print()
print('RETRIEVAL (routing questions only)')
print(f"Hit@1: {df_ret['hit@1'].mean():.0%}  ({df_ret['hit@1'].sum()}/{len(df_ret)})")
print(f"Hit@3: {df_ret['hit@3'].mean():.0%}  ({df_ret['hit@3'].sum()}/{len(df_ret)})")
print(f"Hit@5: {df_ret['hit@5'].mean():.0%}  ({df_ret['hit@5'].sum()}/{len(df_ret)})")
print(f"MRR: {df_ret['mrr'].mean():.4f}")
print(f"nDCG: {df_ret['ndcg'].mean():.4f}")
print()
print('ANSWER QUALITY  (all 20 questions, LLM judge 1-5)')
print(f"Accuracy: {df_judge['accuracy'].mean():.2f}")
print(f"Completeness: {df_judge['completeness'].mean():.2f}")
print(f"Relevance: {df_judge['relevance'].mean():.2f}")
print(f"Overall avg: {df_judge['avg_score'].mean():.2f}")
print()
print('SAFETY')
emerg = df_judge[df_judge['urgency'] == 'emergency']
print(f"Emergency questions: {len(emerg)}")
print(f"Avg score (emergency): {emerg['avg_score'].mean():.2f} / 5")
print()


RAG EVALUATION SUMMARY

RETRIEVAL (routing questions only)
Hit@1: 86%  (12/14)
Hit@3: 100%  (14/14)
Hit@5: 100%  (14/14)
MRR: 0.9286
nDCG: 0.9511

ANSWER QUALITY  (all 20 questions, LLM judge 1-5)
Accuracy: 4.40
Completeness: 4.30
Relevance: 4.95
Overall avg: 4.55

SAFETY
Emergency questions: 2
Avg score (emergency): 4.33 / 5

